# Preprocesamiento de IMDb y MovieLens

Este notebook carga los archivos `.tsv.gz` de `data/imdb` y los `.csv` de `data/mov_lens`, aplica una limpieza base y deja tablas listas para analisis o recomendacion.

## 1. Importar librerias

Nota corta: primero dejamos listas las herramientas de carga y limpieza.

In [1]:
from pathlib import Path

import pandas as pd


## 2. Definir rutas

Nota corta: centralizamos las carpetas para no repetir paths en todo el notebook.

In [2]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'test' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data'
IMDB_DIR = DATA_DIR / 'imdb'
MOVIELENS_DIR = DATA_DIR / 'mov_lens'

IMDB_DIR, MOVIELENS_DIR


(PosixPath('/Users/chperezpelaez/Documents/Github/dmc-tp1-rec-sys/data/imdb'),
 PosixPath('/Users/chperezpelaez/Documents/Github/dmc-tp1-rec-sys/data/mov_lens'))

## 3. Crear funciones de carga

Nota corta: usamos una funcion para estandarizar la lectura y el manejo de valores faltantes.

In [3]:
def load_imdb_table(path: Path) -> pd.DataFrame:
    return pd.read_csv(
        path,
        sep='\t',
        compression='gzip',
        na_values='\\N',
        keep_default_na=True,
        low_memory=False,
    )


def load_movielens_table(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, low_memory=False)


## 4. Cargar todos los archivos

Nota corta: aca levantamos cada archivo y los guardamos en diccionarios para acceder facil por nombre.

In [4]:
imdb_tables = {path.stem.replace('.tsv', ''): load_imdb_table(path) for path in sorted(IMDB_DIR.glob('*.tsv.gz'))}
movielens_tables = {path.stem: load_movielens_table(path) for path in sorted(MOVIELENS_DIR.glob('*.csv'))}

print('Tablas IMDb:', list(imdb_tables))
print('Tablas MovieLens:', list(movielens_tables))


Tablas IMDb: ['name.basics', 'title.akas', 'title.basics', 'title.crew', 'title.episode', 'title.principals', 'title.ratings']
Tablas MovieLens: ['links', 'movies', 'ratings', 'tags']


In [16]:
imdb_tables['title.ratings']

,tconst,averageRating,numVotes
0,tt0000001,5.7,2207
1,tt0000002,5.5,314
2,tt0000003,6.4,2318
3,tt0000004,5.1,198
4,tt0000005,6.2,3044
...,...,...,...
1662792,tt9916846,5.3,7
1662793,tt9916848,5.2,7
1662794,tt9916850,6.0,7
1662795,tt9916852,5.7,7


In [17]:
movielens_tables['links']

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0
...,...,...,...
87580,292731,26812510,1032473.0
87581,292737,14907358,986674.0
87582,292753,12388280,948139.0
87583,292755,64027,182776.0


## 5. Revisar dimensiones iniciales

Nota corta: una vista rapida ayuda a detectar tablas muy grandes o columnas inesperadas.

In [5]:
summary_rows = []

for source, tables in [('IMDb', imdb_tables), ('MovieLens', movielens_tables)]:
    for name, df in tables.items():
        summary_rows.append({
            'source': source,
            'table': name,
            'rows': len(df),
            'columns': df.shape[1],
        })

summary_df = pd.DataFrame(summary_rows).sort_values(['source', 'table']).reset_index(drop=True)
summary_df


,source,table,rows,columns
0,IMDb,name.basics,15257687,6
1,IMDb,title.akas,55865762,8
2,IMDb,title.basics,12446644,9
3,IMDb,title.crew,12446644,3
4,IMDb,title.episode,9612089,4
5,IMDb,title.principals,99029989,6
6,IMDb,title.ratings,1662797,3
7,MovieLens,links,87585,3
8,MovieLens,movies,87585,3
9,MovieLens,ratings,32000204,4


## 6. Limpiar tablas de IMDb

Nota corta: convertimos tipos, quitamos duplicados y separamos columnas multivaluadas cuando conviene.

In [6]:
imdb = {name: df.copy() for name, df in imdb_tables.items()}

# title.basics
imdb['title.basics'] = imdb['title.basics'].drop_duplicates(subset='tconst')
imdb['title.basics']['isAdult'] = imdb['title.basics']['isAdult'].astype('Int64')
for col in ['startYear', 'endYear', 'runtimeMinutes']:
    imdb['title.basics'][col] = pd.to_numeric(imdb['title.basics'][col], errors='coerce').astype('Int64')
imdb['title.basics']['genres_list'] = imdb['title.basics']['genres'].fillna('').apply(
    lambda x: [] if x == '' else x.split(',')
)

# title.ratings
imdb['title.ratings'] = imdb['title.ratings'].drop_duplicates(subset='tconst')
imdb['title.ratings']['averageRating'] = pd.to_numeric(imdb['title.ratings']['averageRating'], errors='coerce')
imdb['title.ratings']['numVotes'] = pd.to_numeric(imdb['title.ratings']['numVotes'], errors='coerce').astype('Int64')

# title.crew
imdb['title.crew'] = imdb['title.crew'].drop_duplicates(subset='tconst')
for col in ['directors', 'writers']:
    imdb['title.crew'][f'{col}_list'] = imdb['title.crew'][col].fillna('').apply(
        lambda x: [] if x == '' else x.split(',')
    )

# title.episode
for col in ['seasonNumber', 'episodeNumber']:
    imdb['title.episode'][col] = pd.to_numeric(imdb['title.episode'][col], errors='coerce').astype('Int64')

# title.principals
imdb['title.principals']['ordering'] = pd.to_numeric(
    imdb['title.principals']['ordering'], errors='coerce'
).astype('Int64')
imdb['title.principals'] = imdb['title.principals'].drop_duplicates()

# title.akas
imdb['title.akas']['ordering'] = pd.to_numeric(imdb['title.akas']['ordering'], errors='coerce').astype('Int64')
imdb['title.akas']['isOriginalTitle'] = pd.to_numeric(
    imdb['title.akas']['isOriginalTitle'], errors='coerce'
).astype('Int64')
imdb['title.akas'] = imdb['title.akas'].drop_duplicates()
for col in ['types', 'attributes']:
    imdb['title.akas'][f'{col}_list'] = imdb['title.akas'][col].fillna('').apply(
        lambda x: [] if x == '' else x.split(',')
    )

# name.basics
imdb['name.basics'] = imdb['name.basics'].drop_duplicates(subset='nconst')
for col in ['birthYear', 'deathYear']:
    imdb['name.basics'][col] = pd.to_numeric(imdb['name.basics'][col], errors='coerce').astype('Int64')
for col in ['primaryProfession', 'knownForTitles']:
    imdb['name.basics'][f'{col}_list'] = imdb['name.basics'][col].fillna('').apply(
        lambda x: [] if x == '' else x.split(',')
    )


## 7. Limpiar tablas de MovieLens

Nota corta: convertimos ids y timestamps, y normalizamos generos y tags.

In [7]:
movielens = {name: df.copy() for name, df in movielens_tables.items()}

# movies
movielens['movies'] = movielens['movies'].drop_duplicates(subset='movieId')
movielens['movies']['movieId'] = pd.to_numeric(movielens['movies']['movieId'], errors='coerce').astype('Int64')
movielens['movies']['genres_list'] = movielens['movies']['genres'].fillna('').replace('(no genres listed)', '').apply(
    lambda x: [] if x == '' else x.split('|')
)
movielens['movies']['release_year'] = movielens['movies']['title'].str.extract(r'\((\d{4})\)$')[0].astype('Int64')
movielens['movies']['clean_title'] = movielens['movies']['title'].str.replace(r'\s*\(\d{4}\)$', '', regex=True)

# ratings
for col in ['userId', 'movieId']:
    movielens['ratings'][col] = pd.to_numeric(movielens['ratings'][col], errors='coerce').astype('Int64')
movielens['ratings']['rating'] = pd.to_numeric(movielens['ratings']['rating'], errors='coerce')
movielens['ratings']['timestamp'] = pd.to_datetime(movielens['ratings']['timestamp'], unit='s', utc=True)
movielens['ratings'] = movielens['ratings'].drop_duplicates()

# links
for col in ['movieId', 'imdbId', 'tmdbId']:
    movielens['links'][col] = pd.to_numeric(movielens['links'][col], errors='coerce').astype('Int64')
movielens['links'] = movielens['links'].drop_duplicates(subset='movieId')
movielens['links']['tconst'] = movielens['links']['imdbId'].apply(
    lambda x: pd.NA if pd.isna(x) else f'tt{int(x):07d}'
)

# tags
for col in ['userId', 'movieId']:
    movielens['tags'][col] = pd.to_numeric(movielens['tags'][col], errors='coerce').astype('Int64')
movielens['tags']['tag'] = movielens['tags']['tag'].astype('string').str.strip()
movielens['tags']['timestamp'] = pd.to_datetime(movielens['tags']['timestamp'], unit='s', utc=True)
movielens['tags'] = movielens['tags'].drop_duplicates()


In [10]:
imdb.keys()

dict_keys(['name.basics', 'title.akas', 'title.basics', 'title.crew', 'title.episode', 'title.principals', 'title.ratings'])

In [11]:
movielens.keys()

dict_keys(['links', 'movies', 'ratings', 'tags'])

## 8. Unir MovieLens con IMDb

Nota corta: usamos `links.csv` para conectar `movieId` con `tconst` y sumar metadatos de IMDb.

In [18]:
movies_enriched = (
    movielens['movies']
    .merge(movielens['links'][['movieId', 'tconst', 'tmdbId']], on='movieId', how='left')
    .merge(
        imdb['title.basics'][['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'runtimeMinutes', 'genres_list']],
        on='tconst',
        how='left',
        suffixes=('_movielens', '_imdb')
    )
    .merge(
        imdb['title.ratings'][['tconst', 'averageRating', 'numVotes']],
        on='tconst',
        how='left'
    )
)

movies_enriched.head()

,movieId,title,genres,genres_list_movielens,release_year,clean_title,tconst,tmdbId,titleType,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres_list_imdb,averageRating,numVotes
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,"[Adventure, Animation, Children, Comedy, Fantasy]",1995,Toy Story,tt0114709,862,movie,Toy Story,Toy Story,0,1995,81,"[Adventure, Animation, Comedy]",8.3,1171117
1,2,Jumanji (1995),Adventure|Children|Fantasy,"[Adventure, Children, Fantasy]",1995,Jumanji,tt0113497,8844,movie,Jumanji,Jumanji,0,1995,104,"[Adventure, Comedy, Family]",7.1,412163
2,3,Grumpier Old Men (1995),Comedy|Romance,"[Comedy, Romance]",1995,Grumpier Old Men,tt0113228,15602,movie,Grumpier Old Men,Grumpier Old Men,0,1995,101,"[Comedy, Romance]",6.7,31652
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,"[Comedy, Drama, Romance]",1995,Waiting to Exhale,tt0114885,31357,movie,Waiting to Exhale,Waiting to Exhale,0,1995,124,"[Comedy, Drama, Romance]",6.0,13798
4,5,Father of the Bride Part II (1995),Comedy,[Comedy],1995,Father of the Bride Part II,tt0113041,11862,movie,Father of the Bride Part II,Father of the Bride Part II,0,1995,106,"[Comedy, Family, Romance]",6.1,45070


In [14]:
movielens['links']

,movieId,imdbId,tmdbId,tconst
0,1,114709,862,tt0114709
1,2,113497,8844,tt0113497
2,3,113228,15602,tt0113228
3,4,114885,31357,tt0114885
4,5,113041,11862,tt0113041
...,...,...,...,...
87580,292731,26812510,1032473,tt26812510
87581,292737,14907358,986674,tt14907358
87582,292753,12388280,948139,tt12388280
87583,292755,64027,182776,tt0064027


In [13]:
imdb['title.ratings']

,tconst,averageRating,numVotes
0,tt0000001,5.7,2207
1,tt0000002,5.5,314
2,tt0000003,6.4,2318
3,tt0000004,5.1,198
4,tt0000005,6.2,3044
...,...,...,...
1662792,tt9916846,5.3,7
1662793,tt9916848,5.2,7
1662794,tt9916850,6.0,7
1662795,tt9916852,5.7,7


## 9. Crear tablas base para recomendacion

Nota corta: dejamos una vista de interacciones y otra de items para usar en el sistema recomendador.

In [19]:
interactions = movielens['ratings'][['userId', 'movieId', 'rating', 'timestamp']].copy()
items = movies_enriched.copy()

print('Interacciones:', interactions.shape)
print('Items:', items.shape)


Interacciones: (32000204, 4)
Items: (87585, 17)


In [20]:
interactions

,userId,movieId,rating,timestamp
0,1,17,4.0,1999-12-03 19:24:37+00:00
1,1,25,1.0,1999-12-03 19:43:48+00:00
2,1,29,2.0,1999-11-22 00:36:16+00:00
3,1,30,5.0,1999-12-03 19:24:37+00:00
4,1,32,5.0,1999-11-22 00:00:58+00:00
...,...,...,...,...
32000199,200948,79702,4.5,2011-01-07 15:03:09+00:00
32000200,200948,79796,1.0,2010-10-16 08:04:52+00:00
32000201,200948,80350,0.5,2011-01-07 15:04:31+00:00
32000202,200948,80463,3.5,2012-10-16 21:43:20+00:00


In [21]:
items

,movieId,title,genres,genres_list_movielens,release_year,clean_title,tconst,tmdbId,titleType,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres_list_imdb,averageRating,numVotes
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,"[Adventure, Animation, Children, Comedy, Fantasy]",1995,Toy Story,tt0114709,862,movie,Toy Story,Toy Story,0,1995,81,"[Adventure, Animation, Comedy]",8.3,1171117
1,2,Jumanji (1995),Adventure|Children|Fantasy,"[Adventure, Children, Fantasy]",1995,Jumanji,tt0113497,8844,movie,Jumanji,Jumanji,0,1995,104,"[Adventure, Comedy, Family]",7.1,412163
2,3,Grumpier Old Men (1995),Comedy|Romance,"[Comedy, Romance]",1995,Grumpier Old Men,tt0113228,15602,movie,Grumpier Old Men,Grumpier Old Men,0,1995,101,"[Comedy, Romance]",6.7,31652
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,"[Comedy, Drama, Romance]",1995,Waiting to Exhale,tt0114885,31357,movie,Waiting to Exhale,Waiting to Exhale,0,1995,124,"[Comedy, Drama, Romance]",6.0,13798
4,5,Father of the Bride Part II (1995),Comedy,[Comedy],1995,Father of the Bride Part II,tt0113041,11862,movie,Father of the Bride Part II,Father of the Bride Part II,0,1995,106,"[Comedy, Family, Romance]",6.1,45070
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87580,292731,The Monroy Affaire (2022),Drama,[Drama],2022,The Monroy Affaire,tt26812510,1032473,movie,El caso Monroy,El caso Monroy,0,2022,95,[Drama],6.9,59
87581,292737,Shelter in Solitude (2023),Comedy|Drama,"[Comedy, Drama]",2023,Shelter in Solitude,tt14907358,986674,movie,Shelter in Solitude,Shelter in Solitude,0,2023,93,"[Comedy, Crime, Drama]",6.2,409
87582,292753,Orca (2023),Drama,[Drama],2023,Orca,tt12388280,948139,movie,Orca,Orca,0,2021,107,[Drama],6.5,781
87583,292755,The Angry Breed (1968),Drama,[Drama],1968,The Angry Breed,tt0064027,182776,movie,The Angry Breed,The Angry Breed,0,1968,94,[Drama],4.8,120


## 10. Validar faltantes y calidad basica

Nota corta: este chequeo final ayuda a ver que tan completa quedo la union entre datasets.

In [22]:
quality_report = pd.DataFrame({
    'missing_tconst': [items['tconst'].isna().mean()],
    'missing_imdb_title': [items['primaryTitle'].isna().mean()],
    'missing_imdb_rating': [items['averageRating'].isna().mean()],
    'missing_tmdbId': [items['tmdbId'].isna().mean()],
})

quality_report.T.rename(columns={0: 'missing_ratio'})


,missing_ratio
missing_tconst,0.000000
missing_imdb_title,0.002580
missing_imdb_rating,0.004133
missing_tmdbId,0.001416


## 11. Vista rapida de resultados

Nota corta: cerramos inspeccionando algunas filas ya limpias y vinculadas.

In [23]:
display(items.head())
display(interactions.head())


,movieId,title,genres,genres_list_movielens,release_year,clean_title,tconst,tmdbId,titleType,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres_list_imdb,averageRating,numVotes
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,"[Adventure, Animation, Children, Comedy, Fantasy]",1995,Toy Story,tt0114709,862,movie,Toy Story,Toy Story,0,1995,81,"[Adventure, Animation, Comedy]",8.3,1171117
1,2,Jumanji (1995),Adventure|Children|Fantasy,"[Adventure, Children, Fantasy]",1995,Jumanji,tt0113497,8844,movie,Jumanji,Jumanji,0,1995,104,"[Adventure, Comedy, Family]",7.1,412163
2,3,Grumpier Old Men (1995),Comedy|Romance,"[Comedy, Romance]",1995,Grumpier Old Men,tt0113228,15602,movie,Grumpier Old Men,Grumpier Old Men,0,1995,101,"[Comedy, Romance]",6.7,31652
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,"[Comedy, Drama, Romance]",1995,Waiting to Exhale,tt0114885,31357,movie,Waiting to Exhale,Waiting to Exhale,0,1995,124,"[Comedy, Drama, Romance]",6.0,13798
4,5,Father of the Bride Part II (1995),Comedy,[Comedy],1995,Father of the Bride Part II,tt0113041,11862,movie,Father of the Bride Part II,Father of the Bride Part II,0,1995,106,"[Comedy, Family, Romance]",6.1,45070


,userId,movieId,rating,timestamp
0,1,17,4.0,1999-12-03 19:24:37+00:00
1,1,25,1.0,1999-12-03 19:43:48+00:00
2,1,29,2.0,1999-11-22 00:36:16+00:00
3,1,30,5.0,1999-12-03 19:24:37+00:00
4,1,32,5.0,1999-11-22 00:00:58+00:00
